# **Text Classification**

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Sample Dataset
texts = ['I loved this movie!', 'It was a terrible movie.']
labels = np.array([1, 0])  # <-- Convert to NumPy array

# Tokenize
tokenizer = Tokenizer(num_words=10000)
tokenizer.fit_on_texts(texts)
x = tokenizer.texts_to_sequences(texts)
x = pad_sequences(x,padding='post')

# Build model
model = Sequential()
model.add(Embedding(input_dim=10000,output_dim=128,input_length=x.shape[1]))
model.add(GRU(units=64))
model.add(Dense(1,activation='sigmoid'))

# Compile
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])

# Train
model.fit(x,labels,epochs=5)

# Predict new review
new_review = ['I hated the movie.']
new_review_seq = tokenizer.texts_to_sequences(new_review)
new_review_seq = pad_sequences(new_review_seq)
prediction = model.predict(new_review_seq)
print("Sentiment:","Positive" if prediction[0][0] > 0.5 else "Negative")

Epoch 1/5


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.0000e+00 - loss: 0.7034
Epoch 2/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step - accuracy: 0.5000 - loss: 0.6931
Epoch 3/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 1.0000 - loss: 0.6830
Epoch 4/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - accuracy: 1.0000 - loss: 0.6728
Epoch 5/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - accuracy: 1.0000 - loss: 0.6624
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 199ms/step
Sentiment: Positive


# **NER**

In [4]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dropout, Dense, TimeDistributed
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Sample sentence and labels
texts = [['Barack', 'Obama', 'was', 'born', 'in', 'Hawaii']]
labels = [['B-PER', 'I-PER', 'O', 'O', 'O', 'B-LOC']]

# Tokenize
tokenizer = Tokenizer(num_words=10000)
tokenizer.fit_on_texts(texts)
x = tokenizer.texts_to_sequences(texts)
x = pad_sequences(x,padding='post')

# Label Mapping
label_map = {'O': 0, 'B-PER': 1, 'I-PER': 2, 'B-LOC': 3}
y = [[label_map[label] for label in sent] for sent in labels]
y = pad_sequences(y,padding='post')
y = np.expand_dims(y,-1)

# Build model
model = Sequential()
model.add(Embedding(input_dim=len(tokenizer.word_index)+1,output_dim=128,input_length=x.shape[1]))
model.add(GRU(units=64,return_sequences=True))
model.add(Dropout(0.1))
model.add(TimeDistributed(Dense(len(label_map),activation='softmax')))

# Compile
model.compile(loss='sparse_categorical_crossentropy',optimizer='adam',metrics=['accuracy'])

# Train
model.fit(x,y,epochs=5)

# Predict on new sentence
new_sentence = ['Obama', 'visited', 'Germany']
new_sentence_seq = tokenizer.texts_to_sequences([new_sentence])
new_sentence_seq = pad_sequences(new_sentence_seq, padding='post', maxlen=x.shape[1])
predictions = model.predict(new_sentence_seq)

# Convert predictions to label names
predicted_labels = [list(label_map.keys())[np.argmax(pred)] for pred in predictions[0][:len(new_sentence)]]
print("Entities:", list(zip(new_sentence, predicted_labels)))

Epoch 1/5


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.5000 - loss: 1.3695
Epoch 2/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.6667 - loss: 1.3505
Epoch 3/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.6667 - loss: 1.3359
Epoch 4/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.6667 - loss: 1.3200
Epoch 5/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - accuracy: 0.6667 - loss: 1.2963
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 166ms/step
Entities: [('Obama', 'O'), ('visited', 'O'), ('Germany', 'O')]
